In [1]:
dataset_info_path ="/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/CVPR25_TextSegFMData_with_class.json"
import json
import os

data = json.load(open(dataset_info_path))

In [6]:
classes = {}
for ds in data:
    classes[ds] = [data[ds][k][0] for k in data[ds] if "ins" not in k] 


In [11]:
# Combine all the values into a single list
all_classes = []
for class_list in classes.values():
    all_classes.extend(class_list)
print(f"There are total {len(all_classes)} classes and {len(set(all_classes))} unique classes")
unique_classes = set(all_classes)

There are total 296 classes and 172 unique classes


In [12]:
unique_classes

{'Adrenocortical carcinoma',
 'Airway Tree',
 "Alzheimer's disease plaque",
 'Aorta',
 'Aortic vessel trees',
 'Arytenoids delineation',
 'Bladder',
 'Brachiocephalic trunk',
 'Brain',
 'Brainstem',
 'Buccal mucosa',
 'COVID-19 infection',
 'Cervical cancer tumor',
 'Cervical esophagus',
 'Colon',
 'Colon cancer primaries',
 'Cricopharyngeal inlet',
 'Duodenum',
 'Endolysosomes',
 'Enhancing tissue',
 'Esophagus',
 'Extra-meatal region of vestibular schwannoma',
 'GTVp and GTVn tumor',
 'Gallbladder',
 'Gastrocnemius Lateralis',
 'Gastrocnemius Medialis',
 'Head-neck cancer',
 'Heart',
 'Inferior vena cava',
 'Intervertebral discs',
 'Intra-meatal region of vestibular schwannoma',
 'Kidney lesions',
 'Larynx-glottis',
 'Larynx-supraglottic',
 'Left Atrium',
 'Left Ventricle',
 'Left adrenal gland',
 'Left anterior segment of the eyeball',
 'Left atrial appendage',
 'Left atrium',
 'Left autochthon',
 'Left brachiocephalic vein',
 'Left carotid artery',
 'Left clavicula',
 'Left cochlea

## New sampling

In [16]:
import os
import json
import pickle
import numpy as np
import random
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

# ========== CONFIGURATION ==========
BASE_DIR = Path("/scratch/railabs/ld258/dataset/public_dataset/CVPR_seg_2025/3D_train_npz_all/")
METADATA_JSON = Path("/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/CVPR25_TextSegFMData_with_class.json")
TMP_DIR = Path("/home/yb107/cvit-work/cvpr2025/SAT/notebook/tmp")

TARGET_FRACTION = 0.1  # 10%
MIN_PER_DATASET = 5
MIN_PER_MODALITY = 50
RANDOM_SEED = 42
N_WORKERS        = os.cpu_count()  

print(f"Using {N_WORKERS} workers for parallel processing")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ensure tmp directory exists
TMP_DIR.mkdir(parents=True, exist_ok=True)


# ========== STEP 1: LOAD METADATA ==========
with METADATA_JSON.open() as f:
    raw_meta = json.load(f)

metadata = {}
for ds_name, info in raw_meta.items():
    label_ids = sorted(int(k) for k in info.keys() if k.isdigit())
    instance_flag = bool(info.get("instance_label", 0))
    metadata[ds_name] = {
        "label_ids": label_ids,
        "instance_label": instance_flag
    }
print("Step 1 Done: metadata loaded")


# ========== STEP 2: DISCOVER ALL CANDIDATE FILES (with checkpoint) ==========
files_by_modality_path = TMP_DIR / "files_by_modality.pkl"
if files_by_modality_path.exists():
    with files_by_modality_path.open("rb") as f:
        files_by_modality = pickle.load(f)
    print("Loaded Step 2 checkpoint: files_by_modality")
else:
    files_by_modality = defaultdict(lambda: defaultdict(list))
    for modality_dir in BASE_DIR.iterdir():
        if not modality_dir.is_dir():
            continue
        modality = modality_dir.name
        for ds_dir in modality_dir.iterdir():
            if not ds_dir.is_dir():
                continue
            ds_name = ds_dir.name
            if ds_name not in metadata:
                print(f"Warning: {ds_name} not in metadata, skipping")
                continue
            for npz_path in ds_dir.glob("*.npz"):
                files_by_modality[modality][ds_name].append(npz_path)
    with files_by_modality_path.open("wb") as f:
        files_by_modality = {k: dict(v) for k, v in files_by_modality.items()}
        pickle.dump(files_by_modality, f)
    print("Step 2 Done and checkpoint saved")


# ========== STEP 3: PRECOMPUTE WEIGHTS AND LABEL EXISTENCE (with checkpoint & corrupted skip) ==========
file_info_path = TMP_DIR / "file_info.pkl"
if file_info_path.exists():
    with file_info_path.open("rb") as f:
        file_info = pickle.load(f)
    total_candidates = len(file_info)
    target_n = int(total_candidates * TARGET_FRACTION)
    print(f"Loaded Step 3 checkpoint: total candidates={total_candidates}, target={target_n}")
else:
    # --- helper for multiprocessing ---------------------------------
    def _process_file(args):
        """Return (ok, modality, ds, path, exist_list, weight)."""
        modality, ds, path, label_ids, inst = args
        try:
            arr = np.load(path)
            gt  = arr["gts"]
            if inst:
                gt = (gt > 0).astype(np.uint8)
            exist = [
                gt.any() if inst else bool((gt == lid).any())
                for lid in label_ids
            ]
            weight = 1 + sum(exist)
            return True, modality, ds, path, exist, weight
        except Exception as e:
            return False, modality, ds, path, None, None
    # ----------------------------------------------------------------

    jobs = []
    for modality, ds_map in files_by_modality.items():
        for ds_name, paths in ds_map.items():
            label_ids     = metadata[ds_name]["label_ids"]
            instance_flag = metadata[ds_name]["instance_label"]
            for p in paths:
                jobs.append((modality, ds_name, p, label_ids, instance_flag))

    file_info     = {}
    corrupted_cnt = 0
    valid_paths   = defaultdict(lambda: defaultdict(list))

    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for ok, modality, ds_name, p, exist, weight in pool.map(_process_file, jobs):
            if ok:
                file_info[(modality, ds_name, p)] = {"exist": exist, "weight": weight}
                valid_paths[modality][ds_name].append(p)
            else:
                corrupted_cnt += 1

    # overwrite files_by_modality with cleaned lists
    files_by_modality = valid_paths
    # save both cleaned structures
    # Convert defaultdict to regular dict for serialization
    files_by_modality = {k: dict(v) for k, v in files_by_modality.items()}
    
    pickle.dump(file_info,        file_info_path.open("wb"))
    pickle.dump(files_by_modality, files_by_modality_path.open("wb"))

    total_candidates = len(file_info)
    target_n         = int(total_candidates * TARGET_FRACTION)
    print(f"Step 3 done: {total_candidates} usable files, {corrupted_cnt} corrupted removed (parallel x{N_WORKERS})")


# ========== STEP 4 & 5: SAMPLING (with checkpoint) ==========
selected_path = TMP_DIR / "selected.pkl"
if selected_path.exists():
    with selected_path.open("rb") as f:
        selected = pickle.load(f)
    print(f"Loaded sampling checkpoint: {len(selected)} entries selected")
else:
    selected = set()

    # Phase 1a: Ensure MIN_PER_DATASET per dataset
    for modality, ds_map in files_by_modality.items():
        for ds_name, paths in ds_map.items():
            candidates = [
                key for key in file_info
                if key[0] == modality and key[1] == ds_name
            ]
            if not candidates:
                continue
            k = min(MIN_PER_DATASET, len(candidates))
            weights = np.array([file_info[key]["weight"] for key in candidates], dtype=float)
            probs = weights / weights.sum()
            chosen = np.random.choice(len(candidates), size=k, replace=False, p=probs)
            for idx in chosen:
                selected.add(candidates[idx])

    # Phase 1b: Ensure MIN_PER_MODALITY per modality
    for modality, ds_map in files_by_modality.items():
        already = [key for key in selected if key[0] == modality]
        need = max(0, MIN_PER_MODALITY - len(already))
        if need == 0:
            continue
        remaining = [
            key for key in file_info
            if key[0] == modality and key not in selected
        ]
        weights = np.array([file_info[key]["weight"] for key in remaining], dtype=float)
        probs = weights / weights.sum()
        chosen = np.random.choice(
            len(remaining), size=min(need, len(remaining)),
            replace=False, p=probs
        )
        for idx in chosen:
            selected.add(remaining[idx])

    # Phase 1c: Prune if overshoot
    if len(selected) > target_n:
        to_remove = len(selected) - target_n
        ds_sizes = {
            ds: len(files_by_modality[mod][ds])
            for mod, ds_map in files_by_modality.items() for ds in ds_map
        }
        selected_list = list(selected)
        rem_weights = np.array([ds_sizes[key[1]] for key in selected_list], dtype=float)
        rem_probs = rem_weights / rem_weights.sum()
        remove_idxs = np.random.choice(
            len(selected_list), size=to_remove,
            replace=False, p=rem_probs
        )
        for idx in remove_idxs:
            selected.remove(selected_list[idx])

    # Phase 2: Fill to target
    remaining_slots = target_n - len(selected)
    if remaining_slots > 0:
        pool = [key for key in file_info if key not in selected]
        weights = np.array([file_info[key]["weight"] for key in pool], dtype=float)
        probs = weights / weights.sum()
        chosen = np.random.choice(
            len(pool), size=min(remaining_slots, len(pool)),
            replace=False, p=probs
        )
        for idx in chosen:
            selected.add(pool[idx])

    with selected_path.open("wb") as f:
        pickle.dump(selected, f)
    print(f"Sampling done: {len(selected)} entries selected. Checkpoint saved")


# ========== STEP 6: WRITE OUTPUT JSONL ==========
out_path = Path(
    "/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/custom_10percent_sample.jsonl"
)
with out_path.open("w") as out_f:
    for modality, ds_name, p in sorted(selected, key=lambda x: (x[0], x[1], str(x[2]))):
        info = file_info[(modality, ds_name, p)]
        record = {
            "data": str(p),
            "dataset": ds_name,
            "modality": modality,
            "label_existance": info["exist"]
        }
        out_f.write(json.dumps(record) + "\n")

print(f"Wrote {len(selected)} entries to {out_path.resolve()}")


Using 48 workers for parallel processing
Step 1 Done: metadata loaded
Step 2 Done and checkpoint saved


KeyboardInterrupt: 

In [4]:
# Cell 1: Imports and helper function
import json
from pathlib import Path

def split_by_modality(input_path: Path, outdir: Path):
    """
    Reads a JSONL file where each line has a "modality" key,
    and writes one JSONL file per modality under outdir/<modality>.jsonl.
    """
    outdir.mkdir(parents=True, exist_ok=True)
    writers = {}
    with input_path.open("r") as infile:
        for line in infile:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            obj['data'] = obj['data'].replace("/scratch/railabs/ld258/dataset/public_dataset/","/cachedata/yb107/")  # Ensure data is a string
            mod = obj.get("modality")
            if mod is None:
                continue  # or handle error

            if mod not in writers:
                writers[mod] = (outdir / f"{mod}.jsonl").open("w")
            writers[mod].write(json.dumps(obj) + "\n")

    for w in writers.values():
        w.close()
    print(f"Split `{input_path}` into {len(writers)} files under `{outdir}`.")

# Cell 2: Parameters — adjust these!
INPUT_JSONL = Path("/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/samples/custom_10percent_sample.jsonl")      # your input JSONL
OUTPUT_DIR  = Path("/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/samples")        # where to put the split files

# run the split
split_by_modality(INPUT_JSONL, OUTPUT_DIR)


Split `/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/samples/custom_10percent_sample.jsonl` into 5 files under `/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/samples`.


In [5]:
import numpy as np

npz_path = "/scratch/railabs/ld258/dataset/public_dataset/CVPR_seg_2025/3D_val_npz/PET_autoPET_fdg_0b98dbe00d_08-11-2002-NA-PET-CT Ganzkoerper  primaer mit KM-83616.npz"
npz = np.load(npz_path, allow_pickle=True)
npz["text_prompts"]

array({'1': 'Lesion delineation in whole body PET imaging', 'instance_label': 1},
      dtype=object)

## Generate `validation_subset.jsonl`

In [2]:
import os
import json
from pathlib import Path
from collections import defaultdict
import random
import numpy as np

# === CONFIGURATION ===
val_npz_dir = "/scratch/railabs/ld258/dataset/public_dataset/CVPR_seg_2025/3D_val_npz"
gt_dir = "/scratch/railabs/ld258/dataset/public_dataset/CVPR_seg_2025/3D_val_gt/3D_val_gt_text"
output_dir = "/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets"  # <- Change this
max_per_modality = {
    "CT": 10,
    "MR": 10,
    "Microscopy": 10,
    "PET": 10,
    "US": 10
}

# === INITIALIZE ===
val_npz_dir = Path(val_npz_dir)
gt_dir = Path(gt_dir)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

modality_files = defaultdict(list)

# === PARSE FILES ===
for file in sorted(val_npz_dir.glob("*.npz")):
    fname = file.name
    parts = fname.split("_")
    modality = parts[0]  # e.g., CT, MR, PET
    dataset = parts[1]  # e.g., autoPET, autoCT, etc.

    # Clean .npz name for consistency
    json_record = {
        "dataset": dataset,
        "modality": modality,
        "img_path": str(file),
        "gt_path": str(gt_dir / fname)
    }
    modality_files[modality].append(json_record)

# === WRITE OUTPUT ===
for modality, entries in modality_files.items():
    max_entries = max_per_modality[modality]
    
    # Shuffle the entries
    random.shuffle(entries)
    
    entries = entries[:max_entries]

    out_path = output_dir / f"{modality.lower()}.jsonl"
    with open(out_path, "w") as f:
        for entry in entries:
            json.dump(entry, f)
            f.write("\n")
    print(f"Wrote {len(entries)} entries to {out_path}")


Wrote 10 entries to /home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets/ct.jsonl
Wrote 10 entries to /home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets/mr.jsonl
Wrote 8 entries to /home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets/microscopy.jsonl
Wrote 10 entries to /home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets/pet.jsonl
Wrote 10 entries to /home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/val_subsets/us.jsonl


In [ ]:
# Cell 1: Imports and parameters (edit as needed)
import os
import json
from collections import defaultdict

# Paths (customize these)
base_dir = "/scratch/railabs/ld258/dataset/public_dataset/CVPR_seg_2025"
val_npz_dir = os.path.join(base_dir, "3D_val_npz")
val_gt_dir = os.path.join(base_dir, "3D_val_gt", "3D_val_gt_text")
metadata_path = "/home/yb107/cvit-work/cvpr2025/SAT/data/dataset_config/cvpr25.json"      # <-- point to your metadata JSON
output_dir = "/home/yb107/cvit-work/cvpr2025/SAT/data/challenge_data/validation_all"             # <-- where ct.jsonl, mr.jsonl, etc. will be written

# Maximum number of cases per modality: either a single int or a dict of modality→int
# e.g. max_cases_per_modality = 50
# or  max_cases_per_modality = {"CT":100, "MR":80, "Microscopy":30, "PET":50, "US":40}
max_cases_per_modality = 10

# Cell 2: Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

# Cell 3: Gather and group files
grouped = defaultdict(list)
for fname in sorted(os.listdir(val_npz_dir)):
    if not fname.endswith(".npz"):
        continue
    img_path = os.path.join(val_npz_dir, fname)
    parts = fname[:-4].split("_")
    modality = parts[0]
    dataset_name = parts[1]  # drop the last part (case identifier)
    
    if "AMOS" in dataset_name:
        dataset_name = "CT_AMOS"
    
    if "COVID" in dataset_name:
        dataset_name = "CT_COVID19-Infection"
    
    if "Low-limb-Leg" in dataset_name:
        dataset_name = "US_Low-limb-Leg"
    
    if "QIN-PROSTATE-Prostate" in dataset_name:
        dataset_name = "MR_QIN-PROSTATE-Lesion"
    
    if "totalseg" in dataset_name.lower():
        dataset_name = fname.split("_s")[0].replace("_mr","")  # e.g., MR_totalseg -> MR
    
    if "heart-ACDC" in dataset_name:
        dataset_name = "MR_Heart_ACDC"   
    
    if "SELMA3D" in dataset_name:
        if "ADplaques" in fname:
            dataset_name = "Microscopy_SELMA3D_ADplaques"
        elif "cfos" in fname:
            # dataset_name = "MR_SELMA3D_cfos"  
            continue
        elif "nuclei" in fname:
            # dataset_name = "MR_SELMA3D_cfos"
            continue
        elif "vessel" in fname:
            dataset_name = "Microscopy_SELMA3D_vessel"
        print(f"___{dataset_name}___{dataset_name in metadata}")
        
    
    if "AMOSMR" in dataset_name:
        dataset_name = "MR_AMOS"
    
    if "LungMasks" in dataset_name:
        dataset_name = "CT_Lungs"
    if "LungLesions" in dataset_name:
        dataset_name = "CT_LungLesion"
        
    if "MR_Head" in fname:
        dataset_name = "MR_HNTS-MRG_HeadTumor"
    
    if "MR_Heart_la" in fname:
        print(f"Dataset '{dataset_name}' not found in metadata. Please update metadata or rename dataset.")
        continue
    
    if "ISLES2022_ADC" in fname:
        dataset_name = "MR_ISLES_ADC"
    if "ISLES2022_DWI" in fname:
        dataset_name = "MR_ISLES_DWI"
    
    if "WMH_FLAIR" in fname:
        dataset_name = "MR_WMH_FLAIR"
    if "WMH_T1" in fname:
        dataset_name = "MR_WMH_T1"
    
    if "US_Cardiac" in fname:
        dataset_name = "US_Cardiac"
    
        
    if dataset_name not in metadata:
        # got through the keys to find the key that contains dataset name
        found = False
        for key in metadata.keys():
            if dataset_name.lower() in key.lower():
                dataset_name = key
                found = True
                break
        if not found:
            if "totalseg" in dataset_name.lower():
                raise ValueError(f"Dataset '{dataset_name}' not found in metadata. Please update metadata or rename dataset.")
            print(f"Dataset '{dataset_name}' not found in metadata. Please update metadata or rename dataset.")
            continue
        
    gt_path = os.path.join(val_gt_dir, fname)
    record = {
        "dataset": dataset_name,
        "modality": modality,
        "img_path": img_path,
        "gt_path": gt_path
    }
    grouped[modality].append(record)

# Cell 4: Write JSONL per modality with optional limit
os.makedirs(output_dir, exist_ok=True)
for modality, records in grouped.items():
    if isinstance(max_cases_per_modality, dict):
        limit = max_cases_per_modality.get(modality, len(records))
    else:
        limit = max_cases_per_modality
    out_path = os.path.join(output_dir, f"{modality.lower()}.jsonl")
    with open(out_path, 'w') as out_f:
        for rec in records[:limit]:
            json.dump(rec, out_f)
            out_f.write("\n")
    
    print(f"Wrote {min(limit, len(records))} records to {out_path}")



Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename dataset.
Dataset 'AbdTumor' not found in metadata. Please update metadata or rename d